# Part 2 — Model Training and Tuning

Trong phần này, ta huấn luyện và chọn siêu tham số cho **Logistic Regression** và **Linear SVM** trên tập Train. TF-IDF được đặt bên trong `Pipeline`, vì vậy mỗi fold tự học từ vựng và IDF chỉ từ phần train của fold đó.

Tập Test được đọc để kiểm tra cấu trúc, nhưng **không tham gia cross-validation, chọn cấu hình hoặc chọn mô hình**.

## 1. Mục tiêu và quy tắc đánh giá

- Dùng 5-fold cross-validation có phân tầng trên `train.csv`.
- Tối ưu **F1 của lớp spam (`label_num = 1`)**.
- Tìm tham số cho TF-IDF (`ngram_range`, `min_df`) và bộ phân loại (`C`).
- Sau khi cả hai lượt tìm kiếm thành công, lưu pipeline tốt nhất của từng mô hình.

`GridSearchCV(refit=True)` tự fit lại cấu hình tốt nhất trên toàn bộ Train sau khi CV hoàn thành.

In [1]:
from pathlib import Path
import sys
import time

import joblib
import pandas as pd
from IPython.display import display
from sklearn.model_selection import GridSearchCV, StratifiedKFold

# Tìm root để notebook chạy từ thư mục dự án hoặc notebooks/.
PROJECT_ROOT = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / "src").is_dir() and (candidate / "data" / "processed").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Không tìm thấy thư mục gốc dự án.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models import TUNING_PARAM_GRID, create_model_pipelines

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
RESULTS_DIR = PROJECT_ROOT / "results"
RANDOM_STATE = 42


## 2. Đọc dữ liệu

Mỗi hàng gồm nội dung email `text` và nhãn `label_num`. Tập Test được nạp và kiểm tra cột/kích thước để xác nhận dữ liệu sẵn sàng cho phần đánh giá sau. Trong Part 2, chỉ `X_train` và `y_train` được đưa vào `GridSearchCV`.

In [2]:
train_path = PROCESSED_DIR / "train.csv"
test_path = PROCESSED_DIR / "test.csv"
if not train_path.is_file() or not test_path.is_file():
    raise FileNotFoundError("Cần chạy Part 1 để tạo train.csv và test.csv trước.")

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
required_columns = {"text", "label_num"}
for name, frame in (("Train", train_df), ("Test", test_df)):
    missing = required_columns - set(frame.columns)
    if missing:
        raise ValueError(f"{name} thiếu cột bắt buộc: {sorted(missing)}")
    if frame["text"].isna().any() or frame["label_num"].isna().any():
        raise ValueError(f"{name} còn giá trị thiếu ở text/label_num.")

X_train = train_df["text"].astype(str)
y_train = train_df["label_num"].astype(int)
print(f"Train dùng cho tuning: {len(train_df):,} hàng")
print(f"Test được giữ riêng cho Part 3: {len(test_df):,} hàng")
print("Phân bố nhãn Train:")
display(y_train.value_counts().sort_index().rename_axis("label_num").to_frame("count"))
print("Test chỉ được nạp/kiểm tra, chưa dùng để fit hoặc chọn tham số.")

Train dùng cho tuning: 3,915 hàng
Test được giữ riêng cho Part 3: 979 hàng
Phân bố nhãn Train:
 count
  2745
  1170
Test chỉ được nạp/kiểm tra, chưa dùng để fit hoặc chọn tham số.


## 3. TF-IDF và các bộ phân loại

TF-IDF chuyển email thành vector trọng số từ/cụm từ. `ngram_range=(1, 1)` dùng unigram; `(1, 2)` dùng unigram và bigram. `min_df` bỏ từ quá hiếm. `C` điều khiển mức regularization của Logistic Regression và Linear SVM.

Cả hai mô hình dùng cùng lưới tham số để so sánh công bằng. Các bước TF-IDF nằm trong cùng Pipeline với classifier nên được fit lại bên trong từng fold, tránh rò rỉ giữa fold huấn luyện và validation.

In [3]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
model_pipelines = create_model_pipelines()
searches = {}
search_rows = []

for model_name, pipeline in model_pipelines.items():
    search = GridSearchCV(
        estimator=pipeline,
        param_grid=TUNING_PARAM_GRID,
        scoring="f1",  # binary F1; label 1 (Spam) là positive class
        cv=cv,
        refit=True,
        n_jobs=1,
        return_train_score=False,
        verbose=1,
        error_score="raise",
    )
    started = time.time()
    print(f"\n===== Tuning {model_name} =====")
    search.fit(X_train, y_train)  # Chỉ fit trên Train; Test không được truyền vào.
    elapsed = time.time() - started
    searches[model_name] = search
    search_rows.append({
        "Model": model_name,
        "Best CV F1 (Spam)": search.best_score_,
        "Best parameters": search.best_params_,
        "Search seconds": round(elapsed, 1),
    })
    print(f"Best CV F1 (Spam): {search.best_score_:.4f}")
    print("Best parameters:", search.best_params_)
    print(f"Elapsed: {elapsed:.1f} seconds")

summary = pd.DataFrame(search_rows)
print("\nTóm tắt tuning (CV chỉ dùng Train):")
display(summary)


===== Tuning Logistic Regression =====
Fitting 5 folds for each of 12 candidates, totalling 60 fits
Best CV F1 (Spam): 0.9805
Best parameters: {'classifier__C': 10.0, 'tfidf__min_df': 2, 'tfidf__ngram_range': (1, 1)}
Elapsed: 52.3 seconds

===== Tuning Linear SVM =====
Fitting 5 folds for each of 12 candidates, totalling 60 fits
Best CV F1 (Spam): 0.9821
Best parameters: {'classifier__C': 1.0, 'tfidf__min_df': 1, 'tfidf__ngram_range': (1, 2)}
Elapsed: 39.2 seconds

Tóm tắt tuning (CV chỉ dùng Train):
              Model  Best CV F1 (Spam)                                                           Best parameters  Search seconds
Logistic Regression           0.980483 {'classifier__C': 10.0, 'tfidf__min_df': 2, 'tfidf__ngram_range': (1, 1)}            52.3
         Linear SVM           0.982149  {'classifier__C': 1.0, 'tfidf__min_df': 1, 'tfidf__ngram_range': (1, 2)}            39.2


## 4. Lưu pipeline sau khi tuning thành công

Chỉ khi cả hai GridSearchCV hoàn tất, pipeline tốt nhất của mỗi mô hình mới được lưu. `best_estimator_` đã được fit lại trên toàn bộ tập Train theo cấu hình tốt nhất. Lưu tên `*_tuned_pipeline.joblib` để không ghi đè các artifacts cũ.

In [4]:
# Chỉ đến cell này khi cả hai lượt tuning đã chạy xong thành công.
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

artifact_paths = {
    "Logistic Regression": ARTIFACTS_DIR / "lr_tuned_pipeline.joblib",
    "Linear SVM": ARTIFACTS_DIR / "svm_tuned_pipeline.joblib",
}
for model_name, artifact_path in artifact_paths.items():
    joblib.dump(searches[model_name].best_estimator_, artifact_path)
    print(f"Đã lưu {model_name}: {artifact_path}")

summary_path = RESULTS_DIR / "cv_tuning_results.csv"
summary.to_csv(summary_path, index=False, encoding="utf-8")
print(f"Đã lưu bảng tham số và điểm CV: {summary_path}")

Đã lưu Logistic Regression: E:\ML_Classification\artifacts\lr_tuned_pipeline.joblib
Đã lưu Linear SVM: E:\ML_Classification\artifacts\svm_tuned_pipeline.joblib
Đã lưu bảng tham số và điểm CV: E:\ML_Classification\results\cv_tuning_results.csv


## 5. Kết luận Part 2

So sánh `Best CV F1 (Spam)` để nhận xét mô hình nào đạt điểm trung bình 5-fold cao hơn trên Train. Đây là điểm phục vụ chọn mô hình/siêu tham số; kết quả tổng quát cuối cùng sẽ được đánh giá riêng trên Test ở Part 3. Không dùng điểm CV thay cho kết quả Test.